In [0]:
# Retrieve credentials from Databricks Secrets

bootstrap_servers = dbutils.secrets.get(
    scope="kafka-secrets",
    key="bootstrap-servers"
)

api_key = dbutils.secrets.get(
    scope="kafka-secrets",
    key="api-key"
)

api_secret = dbutils.secrets.get(
    scope="kafka-secrets",
    key="api-secret"
)

print("Kafka credentials loaded successfully")

Kafka credentials loaded successfully


In [0]:
kafka_options = {
    "kafka.bootstrap.servers": bootstrap_servers,
    "subscribe": "Orders",
    "startingOffsets": "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{api_key}" password="{api_secret}";'
    )
}

In [0]:
kafka_df = (
    spark.read
    .format("kafka")
    .options(**kafka_options)
    .option("endingOffsets", "latest")
    .load()
)

display(
    kafka_df.selectExpr(
        "CAST(key AS STRING) AS order_id",
        "CAST(value AS STRING) AS order_json",
        "topic",
        "partition",
        "offset"
    )
)

order_id,order_json,topic,partition,offset
ORD-1000,"{""event_id"": ""44280347-ad37-472f-8a0b-7b93ad54b57b"", ""order_id"": ""ORD-1000"", ""customer_id"": ""CUST-37"", ""product_id"": ""PROD-12"", ""quantity"": 2, ""amount"": 2550.65, ""status"": ""PLACED"", ""payment_method"": ""NET_BANKING"", ""event_timestamp"": ""2026-11-16T08:45:16.540553+00:00""}",Orders,0,0
ORD-1008,"{""event_id"": ""42b09ff4-2745-4b91-a771-66b1abf71cf5"", ""order_id"": ""ORD-1008"", ""customer_id"": ""CUST-32"", ""product_id"": ""PROD-9"", ""quantity"": 2, ""amount"": 4389.44, ""status"": ""PLACED"", ""payment_method"": ""UPI"", ""event_timestamp"": ""2026-11-16T08:45:16.540553+00:00""}",Orders,0,1
ORD-1009,"{""event_id"": ""50ebe524-df03-492a-89e5-e2dd0209df52"", ""order_id"": ""ORD-1009"", ""customer_id"": ""CUST-47"", ""product_id"": ""PROD-18"", ""quantity"": 4, ""amount"": 4509.28, ""status"": ""PLACED"", ""payment_method"": ""COD"", ""event_timestamp"": ""2026-11-16T08:45:16.540553+00:00""}",Orders,0,2
ORD-1000,"{""event_id"": ""a8f34bbb-5f3c-4631-a54b-36e7b009fb63"", ""order_id"": ""ORD-1000"", ""customer_id"": ""CUST-8"", ""product_id"": ""PROD-18"", ""quantity"": 1, ""amount"": 1827.62, ""status"": ""PLACED"", ""payment_method"": ""NET_BANKING"", ""event_timestamp"": ""2026-11-16T09:41:52.710442+00:00""}",Orders,0,3
ORD-1008,"{""event_id"": ""e86d6e4b-2f04-476a-8bed-4ffe8fb46caa"", ""order_id"": ""ORD-1008"", ""customer_id"": ""CUST-9"", ""product_id"": ""PROD-1"", ""quantity"": 3, ""amount"": 4218.06, ""status"": ""PLACED"", ""payment_method"": ""NET_BANKING"", ""event_timestamp"": ""2026-11-16T09:41:52.711364+00:00""}",Orders,0,4
ORD-1009,"{""event_id"": ""c4555a4a-123e-44c5-89d5-432617bd34f8"", ""order_id"": ""ORD-1009"", ""customer_id"": ""CUST-49"", ""product_id"": ""PROD-7"", ""quantity"": 1, ""amount"": 2532.44, ""status"": ""PLACED"", ""payment_method"": ""COD"", ""event_timestamp"": ""2026-11-16T09:41:52.711364+00:00""}",Orders,0,5
ORD-1000,"{""event_id"": ""e3ce09ed-45ac-4aa8-818b-fee0b2a38181"", ""order_id"": ""ORD-1000"", ""customer_id"": ""CUST-34"", ""product_id"": ""PROD-19"", ""quantity"": 3, ""amount"": 1524.53, ""status"": ""PLACED"", ""payment_method"": ""CARD"", ""event_timestamp"": ""2026-11-16T09:44:47.505279+00:00""}",Orders,0,6
ORD-1008,"{""event_id"": ""103f281d-affd-4d64-9b32-216807b8518a"", ""order_id"": ""ORD-1008"", ""customer_id"": ""CUST-39"", ""product_id"": ""PROD-4"", ""quantity"": 5, ""amount"": 2453.59, ""status"": ""PLACED"", ""payment_method"": ""COD"", ""event_timestamp"": ""2026-11-16T09:44:47.505279+00:00""}",Orders,0,7
ORD-1009,"{""event_id"": ""e2d56e60-6e72-4bd4-a779-c3b9b6c78246"", ""order_id"": ""ORD-1009"", ""customer_id"": ""CUST-33"", ""product_id"": ""PROD-2"", ""quantity"": 2, ""amount"": 2862.57, ""status"": ""PLACED"", ""payment_method"": ""NET_BANKING"", ""event_timestamp"": ""2026-11-16T09:44:47.505279+00:00""}",Orders,0,8
ORD-1000,"{""event_id"": ""cd06e684-4340-4f84-a8fe-8de3e5783433"", ""order_id"": ""ORD-1000"", ""customer_id"": ""CUST-10"", ""product_id"": ""PROD-2"", ""quantity"": 2, ""amount"": 2233.99, ""status"": ""PLACED"", ""payment_method"": ""NET_BANKING"", ""event_timestamp"": ""2026-11-16T09:51:43.906773+00:00""}",Orders,0,9


In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DoubleType
)

order_schema = StructType([
    StructField("event_id", StringType()),
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType()),
    StructField("quantity", IntegerType()),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
    StructField("payment_method", StringType()),
    StructField("event_timestamp", StringType())
])

parsed_df = kafka_df.select(
    from_json(
        col("value").cast("string"),
        order_schema
    ).alias("data"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafka_timestamp")
).select(
    "data.*",
    "topic",
    "partition",
    "offset",
    "kafka_timestamp"
)

display(parsed_df)

event_id,order_id,customer_id,product_id,quantity,amount,status,payment_method,event_timestamp,topic,partition,offset,kafka_timestamp
44280347-ad37-472f-8a0b-7b93ad54b57b,ORD-1000,CUST-37,PROD-12,2,2550.65,PLACED,NET_BANKING,2026-11-16T08:45:16.540553+00:00,Orders,0,0,2026-11-16T08:45:16.540Z
42b09ff4-2745-4b91-a771-66b1abf71cf5,ORD-1008,CUST-32,PROD-9,2,4389.44,PLACED,UPI,2026-11-16T08:45:16.540553+00:00,Orders,0,1,2026-11-16T08:45:16.540Z
50ebe524-df03-492a-89e5-e2dd0209df52,ORD-1009,CUST-47,PROD-18,4,4509.28,PLACED,COD,2026-11-16T08:45:16.540553+00:00,Orders,0,2,2026-11-16T08:45:16.540Z
a8f34bbb-5f3c-4631-a54b-36e7b009fb63,ORD-1000,CUST-8,PROD-18,1,1827.62,PLACED,NET_BANKING,2026-11-16T09:41:52.710442+00:00,Orders,0,3,2026-11-16T09:41:52.710Z
e86d6e4b-2f04-476a-8bed-4ffe8fb46caa,ORD-1008,CUST-9,PROD-1,3,4218.06,PLACED,NET_BANKING,2026-11-16T09:41:52.711364+00:00,Orders,0,4,2026-11-16T09:41:52.711Z
c4555a4a-123e-44c5-89d5-432617bd34f8,ORD-1009,CUST-49,PROD-7,1,2532.44,PLACED,COD,2026-11-16T09:41:52.711364+00:00,Orders,0,5,2026-11-16T09:41:52.711Z
e3ce09ed-45ac-4aa8-818b-fee0b2a38181,ORD-1000,CUST-34,PROD-19,3,1524.53,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,0,6,2026-11-16T09:44:47.505Z
103f281d-affd-4d64-9b32-216807b8518a,ORD-1008,CUST-39,PROD-4,5,2453.59,PLACED,COD,2026-11-16T09:44:47.505279+00:00,Orders,0,7,2026-11-16T09:44:47.505Z
e2d56e60-6e72-4bd4-a779-c3b9b6c78246,ORD-1009,CUST-33,PROD-2,2,2862.57,PLACED,NET_BANKING,2026-11-16T09:44:47.505279+00:00,Orders,0,8,2026-11-16T09:44:47.505Z
cd06e684-4340-4f84-a8fe-8de3e5783433,ORD-1000,CUST-10,PROD-2,2,2233.99,PLACED,NET_BANKING,2026-11-16T09:51:43.906773+00:00,Orders,0,9,2026-11-16T09:51:43.906Z


In [0]:
kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

In [0]:
parsed_stream_df = (
    kafka_stream_df
    .select(
        from_json(
            col("value").cast("string"),
            order_schema
        ).alias("data"),
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    )
    .select(
        "data.*",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp"
    )
)

In [0]:
%sql
SHOW CATALOGS;

catalog
db_project_m_catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
bronze
default
gold
information_schema
realtime_ecommerce
silver


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.realtime_ecommerce;


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS
workspace.realtime_ecommerce.streaming_checkpoints;

In [0]:
%sql
SHOW VOLUMES IN workspace.realtime_ecommerce;

database,volume_name
realtime_ecommerce,streaming_checkpoints


In [0]:
checkpoint_path = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/kafka_orders"
)

display(
    parsed_stream_df,
    checkpointLocation=checkpoint_path
)

Checkpointing to /Volumes/workspace/realtime_ecommerce/streaming_checkpoints/kafka_orders


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7496784249214066>, line 6
      1 checkpoint_path = (
      2     "/Volumes/workspace/realtime_ecommerce/"
      3     "streaming_checkpoints/kafka_orders"
      4 )
----> 6 display(
      7     parsed_stream_df,
      8     checkpointLocation=checkpoint_path
      9 )

File <command-7496784249214057>, line 2
      1 parsed_stream_df = (
----> 2     kafka_stream_df
      3     .select(
      4         from_json(
      5             col("value").cast("string"),
      6             order_schema
      7         ).alias("data"),
      8         col("topic"),
      9         col("partition"),
     10         col("offset"),
     11         col("timestamp").alias("kafka_timestamp")
     12     )
     13     .select(
     14         "data.*",
     15         "topic",
     16         "partition",
     17         "offset",
     18  

In [0]:
bronze_checkpoint = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/bronze_orders"
)

bronze_query = (
    parsed_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.realtime_ecommerce.bronze_orders")
)

In [0]:
print("Is active:", bronze_query.isActive)
print("Status:", bronze_query.status)
print("Last progress:", bronze_query.lastProgress)

Is active: False
Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Last progress: {
    "id": "4e87e346-01d4-4fa3-b332-959f5d10b212",
    "runId": "b1d3072b-af33-4c15-ba48-7c66d9378ca6",
    "name": null,
    "timestamp": "2026-11-16T09:48:11.028Z",
    "batchId": 0,
    "batchDuration": 4364,
    "durationMs": {
        "triggerExecution": 4363,
        "queryPlanning": 215,
        "collectSourceMetrics": 12,
        "getBatch": 103,
        "commitOffsets": 88,
        "postCommitBatchCallback": 1,
        "addBatch": 2765,
        "latestOffset": 10,
        "postAddBatchCallback": 1,
        "commitBatch": 573,
        "walCommit": 85
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[Orders]]",
            "startOffset": null,
            "endOffset": "{\"Orders\":{\"0\":9,\"1\":12,\"2\":9}}",
            "latestOffset": "{\"Orders\":{\"0\":9,\"1\":12,\"2\":9}}",
         

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.realtime_ecommerce.bronze_orders;

COUNT(*)
120


In [0]:
%sql
SELECT *
FROM workspace.realtime_ecommerce.bronze_orders
ORDER BY kafka_timestamp DESC
LIMIT 10;

event_id,order_id,customer_id,product_id,quantity,amount,status,payment_method,event_timestamp,topic,partition,offset,kafka_timestamp
3565159d-d7b8-4ffc-983b-76a1a7f8199c,ORD-1004,CUST-35,PROD-3,3,2422.5,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,2,8,2026-11-16T09:44:47.505Z
2283109a-63b3-4fbe-bf8b-51d08f69d4dc,ORD-1003,CUST-30,PROD-7,3,904.05,PLACED,COD,2026-11-16T09:44:47.505279+00:00,Orders,2,7,2026-11-16T09:44:47.505Z
357cc240-733e-4f25-bb10-7671ed40b8ad,ORD-1001,CUST-42,PROD-16,4,288.35,PLACED,NET_BANKING,2026-11-16T09:44:47.505279+00:00,Orders,1,8,2026-11-16T09:44:47.505Z
e3ce09ed-45ac-4aa8-818b-fee0b2a38181,ORD-1000,CUST-34,PROD-19,3,1524.53,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,0,6,2026-11-16T09:44:47.505Z
ddb8530b-7a1b-4da4-a730-58f119eeec1a,ORD-1006,CUST-29,PROD-6,3,3274.75,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,1,10,2026-11-16T09:44:47.505Z
9a86b86a-e24a-48d5-8c5f-84018e003602,ORD-1002,CUST-47,PROD-18,2,2319.12,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,2,6,2026-11-16T09:44:47.505Z
fecaefc5-2f58-445d-8513-2eee05cb3846,ORD-1005,CUST-29,PROD-14,5,1796.37,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,1,9,2026-11-16T09:44:47.505Z
f7c63b74-5e6d-4d47-bccb-6109ad1de70f,ORD-1007,CUST-8,PROD-7,5,4413.79,PLACED,CARD,2026-11-16T09:44:47.505279+00:00,Orders,1,11,2026-11-16T09:44:47.505Z
103f281d-affd-4d64-9b32-216807b8518a,ORD-1008,CUST-39,PROD-4,5,2453.59,PLACED,COD,2026-11-16T09:44:47.505279+00:00,Orders,0,7,2026-11-16T09:44:47.505Z
e2d56e60-6e72-4bd4-a779-c3b9b6c78246,ORD-1009,CUST-33,PROD-2,2,2862.57,PLACED,NET_BANKING,2026-11-16T09:44:47.505279+00:00,Orders,0,8,2026-11-16T09:44:47.505Z


In [0]:
bronze_table = "workspace.realtime_ecommerce.bronze_orders"

bronze_stream = (
    spark.readStream
    .table(bronze_table)
)

bronze_stream.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    to_date
)

silver_stream = (
    bronze_stream

    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("quantity", col("quantity").cast("integer"))

    .filter(
        col("event_id").isNotNull() &
        col("order_id").isNotNull() &
        col("amount").isNotNull() &
        col("quantity").isNotNull() &
        col("event_timestamp").isNotNull()
    )

    .withColumn(
        "event_date",
        to_date(col("event_timestamp"))
    )
)

silver_stream.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [0]:
silver_checkpoint = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/silver_orders"
)

silver_query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.realtime_ecommerce.silver_orders")
)

In [0]:
%sql
SELECT COUNT(*) AS silver_records
FROM workspace.realtime_ecommerce.silver_orders;

silver_records
120


In [0]:
%sql
SELECT *
FROM workspace.realtime_ecommerce.silver_orders
LIMIT 10;

event_id,order_id,customer_id,product_id,quantity,amount,status,payment_method,event_timestamp,topic,partition,offset,kafka_timestamp,event_date
5db7f55c-84e4-4e18-9bb2-b4ac163255e0,ORD-1001,CUST-32,PROD-18,5,1485.66,PLACED,COD,2026-11-16T08:45:16.540Z,Orders,1,0,2026-11-16T08:45:16.540Z,2026-11-16
b2da01c3-9d18-4d1a-90fa-a11316deef1a,ORD-1005,CUST-7,PROD-17,3,579.68,PLACED,COD,2026-11-16T08:45:16.540Z,Orders,1,1,2026-11-16T08:45:16.540Z,2026-11-16
55622676-48ac-410c-93ed-3dbdc1627bd6,ORD-1006,CUST-17,PROD-5,1,1726.35,PLACED,UPI,2026-11-16T08:45:16.540Z,Orders,1,2,2026-11-16T08:45:16.540Z,2026-11-16
1492e1e8-cb51-4e59-bbee-0f0f95377456,ORD-1007,CUST-18,PROD-13,4,3443.7,PLACED,UPI,2026-11-16T08:45:16.540Z,Orders,1,3,2026-11-16T08:45:16.540Z,2026-11-16
25c8964a-0be4-47d8-8754-5647dbf3b329,ORD-1001,CUST-24,PROD-2,1,1782.81,PLACED,UPI,2026-11-16T09:41:52.710Z,Orders,1,4,2026-11-16T09:41:52.710Z,2026-11-16
d0b87a25-8d32-4693-8477-ad134c9a4539,ORD-1005,CUST-38,PROD-17,4,4577.33,PLACED,NET_BANKING,2026-11-16T09:41:52.710Z,Orders,1,5,2026-11-16T09:41:52.710Z,2026-11-16
44399641-4a93-480f-80d9-5b4168644f66,ORD-1006,CUST-4,PROD-16,3,2859.19,PLACED,NET_BANKING,2026-11-16T09:41:52.710Z,Orders,1,6,2026-11-16T09:41:52.711Z,2026-11-16
99418a2f-c48d-43d6-95e2-232bf9ad094b,ORD-1007,CUST-12,PROD-16,5,3844.51,PLACED,UPI,2026-11-16T09:41:52.711Z,Orders,1,7,2026-11-16T09:41:52.711Z,2026-11-16
357cc240-733e-4f25-bb10-7671ed40b8ad,ORD-1001,CUST-42,PROD-16,4,288.35,PLACED,NET_BANKING,2026-11-16T09:44:47.505Z,Orders,1,8,2026-11-16T09:44:47.505Z,2026-11-16
fecaefc5-2f58-445d-8513-2eee05cb3846,ORD-1005,CUST-29,PROD-14,5,1796.37,PLACED,CARD,2026-11-16T09:44:47.505Z,Orders,1,9,2026-11-16T09:44:47.505Z,2026-11-16


In [0]:
spark.sql("""
    DROP TABLE IF EXISTS
    workspace.realtime_ecommerce.gold_order_metrics
""")

DataFrame[]

In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    round,
    countDistinct
)

silver_df = spark.read.table(
    "workspace.realtime_ecommerce.silver_orders"
)

gold_order_metrics = (
    silver_df
    .groupBy("status")
    .agg(
        count("event_id").alias("total_events"),
        countDistinct("customer_id").alias("unique_customers"),
        sum("quantity").alias("total_quantity"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("amount"), 2).alias("avg_order_amount")
    )
)

gold_order_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.realtime_ecommerce.gold_order_metrics"
    )

display(gold_order_metrics)

status,total_events,unique_customers,total_quantity,total_revenue,avg_order_amount
DELIVERED,10,10,20,29306.71,2930.67
PLACED,92,45,282,237390.65,2580.33
CANCELLED,4,3,14,13158.44,3289.61
SHIPPED,14,13,38,38248.97,2732.07


In [0]:
from pyspark.sql.functions import sum

silver_count = (
    spark.read.table(
        "workspace.realtime_ecommerce.silver_orders"
    ).count()
)

gold_count = (
    spark.read.table(
        "workspace.realtime_ecommerce.gold_order_metrics"
    )
    .agg(sum("total_events").alias("total"))
    .first()["total"]
)

print("Silver records:", silver_count)
print("Gold events:", gold_count)

if silver_count == gold_count:
    print("Validation: PASSED")
else:
    print("Validation: FAILED")

Silver records: 120
Gold events: 120
Validation: PASSED


In [0]:
print("Bronze:", spark.read.table(
    "workspace.realtime_ecommerce.bronze_orders"
).count())

print("Silver:", spark.read.table(
    "workspace.realtime_ecommerce.silver_orders"
).count())

print("Gold:")
display(spark.read.table(
    "workspace.realtime_ecommerce.gold_order_metrics"
))


Bronze: 120
Silver: 120
Gold:


status,total_events,unique_customers,total_quantity,total_revenue,avg_order_amount
DELIVERED,10,10,20,29306.71,2930.67
PLACED,92,45,282,237390.65,2580.33
CANCELLED,4,3,14,13158.44,3289.61
SHIPPED,14,13,38,38248.97,2732.07


In [0]:
from pyspark.sql.functions import (
    col,
    window,
    sum,
    count,
    to_timestamp
)

silver_stream = (
    spark.readStream
    .table("workspace.realtime_ecommerce.silver_orders")
)

windowed_revenue = (
    silver_stream
    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )
    .groupBy(
        window(col("event_timestamp"), "1 minute")
    )
    .agg(
        count("event_id").alias("total_orders"),
        sum("amount").alias("total_revenue")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("total_orders"),
        col("total_revenue")
    )
)

In [0]:
window_checkpoint = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/windowed_revenue"
)

window_query = (
    windowed_revenue.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", window_checkpoint)
    .trigger(availableNow=True)
    .toTable(
        "workspace.realtime_ecommerce.gold_windowed_revenue"
    )
)

In [0]:
display(
    spark.read.table(
        "workspace.realtime_ecommerce.gold_windowed_revenue"
    ).orderBy("window_start")
)

window_start,window_end,total_orders,total_revenue
2026-11-16T08:45:00.000Z,2026-11-16T08:46:00.000Z,10,26486.91
2026-11-16T09:41:00.000Z,2026-11-16T09:42:00.000Z,10,33004.93
2026-11-16T09:44:00.000Z,2026-11-16T09:45:00.000Z,10,22259.619999999995
2026-11-16T09:51:00.000Z,2026-11-16T09:52:00.000Z,10,26330.100000000002
2026-11-16T10:12:00.000Z,2026-11-16T10:13:00.000Z,10,25603.13


Tumbling Window


In [0]:
from pyspark.sql.functions import (
    col,
    window,
    sum,
    count,
    to_timestamp
)

silver_stream = (
    spark.readStream
    .table("workspace.realtime_ecommerce.silver_orders")
)

watermarked_revenue = (
    silver_stream
    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )
    .withWatermark("event_timestamp", "2 minutes")
    .groupBy(
        window(col("event_timestamp"), "1 minute")
    )
    .agg(
        count("event_id").alias("total_orders"),
        sum("amount").alias("total_revenue")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("total_orders"),
        col("total_revenue")
    )
)

In [0]:
watermark_checkpoint = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/watermark_revenue"
)

watermark_query = (
    watermarked_revenue.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", watermark_checkpoint)
    .trigger(availableNow=True)
    .toTable(
        "workspace.realtime_ecommerce.gold_watermark_revenue"
    )
)

In [0]:
display(
    spark.read.table(
        "workspace.realtime_ecommerce.gold_watermark_revenue"
    ).orderBy("window_start")
)

window_start,window_end,total_orders,total_revenue
2026-11-16T08:45:00.000Z,2026-11-16T08:46:00.000Z,10,26486.91
2026-11-16T09:41:00.000Z,2026-11-16T09:42:00.000Z,10,33004.93
2026-11-16T09:44:00.000Z,2026-11-16T09:45:00.000Z,10,22259.619999999995
2026-11-16T09:51:00.000Z,2026-11-16T09:52:00.000Z,10,26330.100000000002


Sliding Window


In [0]:
from pyspark.sql.functions import (
    col,
    window,
    sum,
    count
)

silver_stream = (
    spark.readStream
    .table("workspace.realtime_ecommerce.silver_orders")
)

sliding_revenue = (
    silver_stream
    .withWatermark("event_timestamp", "2 minutes")
    .groupBy(
        window(
            col("event_timestamp"),
            "10 minutes",
            "5 minutes"
        )
    )
    .agg(
        count("event_id").alias("total_orders"),
        sum("amount").alias("total_revenue")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("total_orders"),
        col("total_revenue")
    )
)

In [0]:
sliding_checkpoint = (
    "/Volumes/workspace/realtime_ecommerce/"
    "streaming_checkpoints/sliding_revenue"
)

sliding_query = (
    sliding_revenue.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", sliding_checkpoint)
    .trigger(availableNow=True)
    .toTable(
        "workspace.realtime_ecommerce.gold_sliding_revenue"
    )
)

In [0]:
display(
    spark.read.table(
        "workspace.realtime_ecommerce.gold_sliding_revenue"
    ).orderBy("window_start")
)

window_start,window_end,total_orders,total_revenue
2026-11-16T08:40:00.000Z,2026-11-16T08:50:00.000Z,10,26486.91
2026-11-16T08:45:00.000Z,2026-11-16T08:55:00.000Z,10,26486.91
2026-11-16T09:35:00.000Z,2026-11-16T09:45:00.000Z,20,55264.55000000001
2026-11-16T09:40:00.000Z,2026-11-16T09:50:00.000Z,20,55264.55000000001
2026-11-16T09:45:00.000Z,2026-11-16T09:55:00.000Z,10,26330.100000000002
2026-11-16T09:50:00.000Z,2026-11-16T10:00:00.000Z,10,26330.100000000002


In [0]:
from pyspark.sql.functions import col

display(
    spark.read.table(
        "workspace.realtime_ecommerce.silver_orders"
    )
    .groupBy("status")
    .count()
    .orderBy(col("count").desc())
)

status,count
PLACED,92
SHIPPED,14
DELIVERED,10
CANCELLED,4


In [0]:
display(
    spark.read.table(
        "workspace.realtime_ecommerce.silver_orders"
    )
    .orderBy(col("event_timestamp").desc())
    .select("event_id", "order_id", "status")
    .limit(10)
)

event_id,order_id,status
2e330307-fe6d-4a20-b7f6-7b4ea8d2cfbb,ORD65CDE6BBFF,DELIVERED
2c6b523c-52b1-40e8-a834-69feed75df0a,ORD26F5C159AA,SHIPPED
c4f07342-c8ab-4d40-9990-0d0f05f3e975,ORDC496FA6732,PLACED
c715dcfc-3e9c-4c48-9bfc-966954d7ee27,ORD91E99C263E,SHIPPED
a96e1b9c-858c-4dd6-b2d9-e5240e20d02b,ORD3DDB900F4C,PLACED
14c243b2-bd92-44c2-8e87-cb5483bb2172,ORD107864E677,PLACED
86dc248e-9612-424e-b578-1e7a9ff41d37,ORD9D9889C493,SHIPPED
b8057bf3-b5f7-4552-90e9-1a8fcc293944,ORD5F11DE2BAB,SHIPPED
48cca361-78de-4685-b7fc-0b20eda6011f,ORD760D0F2112,SHIPPED
e14bdcdf-2fd5-47bd-95cc-819736d587a1,ORDAD40431E56,DELIVERED


In [0]:
display(
    spark.read.table(
        "workspace.realtime_ecommerce.gold_order_metrics"
    )
)


status,total_events,unique_customers,total_quantity,total_revenue,avg_order_amount
DELIVERED,10,10,20,29306.71,2930.67
PLACED,92,45,282,237390.65,2580.33
CANCELLED,4,3,14,13158.44,3289.61
SHIPPED,14,13,38,38248.97,2732.07


In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    round
)

silver_df = spark.read.table(
    "workspace.realtime_ecommerce.silver_orders"
)

gold_payment_metrics = (
    silver_df
    .groupBy("payment_method")
    .agg(
        count("event_id").alias("total_orders"),
        round(sum("amount"), 2).alias("total_revenue"),
        round(avg("amount"), 2).alias("avg_order_amount")
    )
    .orderBy("total_revenue", ascending=False)
)

gold_payment_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.realtime_ecommerce.gold_payment_metrics"
    )

display(gold_payment_metrics)

payment_method,total_orders,total_revenue,avg_order_amount
UPI,32,87755.98,2742.37
NET_BANKING,34,84780.91,2493.56
CARD,27,77452.69,2868.62
COD,27,68115.19,2522.78


In [0]:
from pyspark.sql.functions import sum, count, round

gold_product_metrics = (
    silver_df
    .groupBy("product_id")
    .agg(
        count("event_id").alias("total_orders"),
        sum("quantity").alias("units_sold"),
        round(sum("amount"), 2).alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)

gold_product_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.realtime_ecommerce.gold_product_metrics"
    )

display(gold_product_metrics)

product_id,total_orders,units_sold,total_revenue
PROD-2,10,28,25479.86
PROD-6,8,20,22869.37
PROD-8,7,28,22683.5
PROD-18,8,19,22643.73
PROD-16,7,27,22359.31
PROD-5,10,28,20412.17
PROD-3,7,13,20214.38
PROD-12,7,17,18873.66
PROD-20,6,18,17618.48
PROD-7,7,21,17097.32


In [0]:
from pyspark.sql.functions import sum

silver_count = silver_df.count()

status_count = (
    spark.read.table(
        "workspace.realtime_ecommerce.gold_order_metrics"
    )
    .agg(sum("total_events"))
    .first()[0]
)

payment_count = (
    spark.read.table(
        "workspace.realtime_ecommerce.gold_payment_metrics"
    )
    .agg(sum("total_orders"))
    .first()[0]
)

product_count = (
    spark.read.table(
        "workspace.realtime_ecommerce.gold_product_metrics"
    )
    .agg(sum("total_orders"))
    .first()[0]
)

print("Silver records:", silver_count)
print("Status aggregation:", status_count)
print("Payment aggregation:", payment_count)
print("Product aggregation:", product_count)

print(
    "Validation:",
    "PASSED" if all(
        x == silver_count
        for x in [status_count, payment_count, product_count]
    ) else "FAILED"
)

Silver records: 120
Status aggregation: 120
Payment aggregation: 120
Product aggregation: 120
Validation: PASSED
